In [ ]:
# -*- coding: utf-8 -*-

from pathlib import Path
from datetime import datetime
import time

import pandas as pd
import matplotlib.pyplot as plt


# ==========================
# 設定
# ==========================
BASE_DIR = Path(
    r"D:\musashino-university\finance\change_point_output_advanced"
)

OUTPUT_DIR = BASE_DIR

METHOD_PLOT_DIR = OUTPUT_DIR / "plots_by_method"
RANKING_PLOT_DIR = OUTPUT_DIR / "plots_date_ranking"

OUT_METHOD_CSV = OUTPUT_DIR / "change_point_daily_counts_by_method_all_csv.csv"
OUT_METHOD_OVERLAY_PNG = OUTPUT_DIR / "daily_change_point_by_method_overlay_all_csv.png"

OUT_DATE_RANKING_CSV = OUTPUT_DIR / "daily_total_change_point_ranking_all_csv.csv"
OUT_DATE_RANKING_TOP_PNG = OUTPUT_DIR / "daily_total_change_point_ranking_top30_all_csv.png"

OUT_METHOD_RANKING_CSV = OUTPUT_DIR / "change_point_method_ranking_all_csv.csv"

SAVE_ENCODING = "utf-8-sig"

METHOD_ORDER = ["pelt", "binseg", "bottomup", "dynp", "window"]

FIGSIZE_METHOD_OVERLAY = (18, 8)
FIGSIZE_METHOD_SINGLE = (16, 6)
FIGSIZE_RANKING = (14, 10)

DPI = 150

SHOW_METHOD_OVERLAY = True
SHOW_METHOD_SINGLE = True
SHOW_DATE_RANKING = True

TOP_N_DATES = 30

REQUIRED_COLS = {
    "method",
    "model",
    "combo",
    "date",
    "change_point_count"
}


# ==========================
# ログ
# ==========================
START_TIME = time.time()

def log(msg):
    elapsed = time.time() - START_TIME
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{now} | +{elapsed:,.1f}s] {msg}")


# ==========================
# 準備
# ==========================
METHOD_PLOT_DIR.mkdir(parents=True, exist_ok=True)
RANKING_PLOT_DIR.mkdir(parents=True, exist_ok=True)

log("=" * 70)
log("CSVファイル探索開始")

csv_files = sorted(BASE_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"CSVファイルが見つかりません: {BASE_DIR}")

log(f"検出CSV数: {len(csv_files):,}")


# ==========================
# すべてのCSVを読み込み
# ==========================
dfs = []
skipped_files = []
failed_files = []

log("=" * 70)
log("CSV読み込み開始")

for i, csv_path in enumerate(csv_files, start=1):

    log(f"[{i}/{len(csv_files)}] 読み込み確認: {csv_path.name}")

    try:
        tmp = pd.read_csv(csv_path, encoding=SAVE_ENCODING)

        missing_cols = REQUIRED_COLS - set(tmp.columns)

        if missing_cols:
            log(f"スキップ（必要列なし）: {csv_path.name} / 不足列: {missing_cols}")
            skipped_files.append(csv_path.name)
            continue

        tmp["source_csv"] = csv_path.name
        dfs.append(tmp)

        log(f"読み込み成功: {csv_path.name} / 行数: {len(tmp):,}")

    except Exception as e:
        log(f"読み込み失敗: {csv_path.name} / {e}")
        failed_files.append(csv_path.name)


if not dfs:
    raise ValueError("有効なCSVがありません。必要列を持つCSVが見つかりませんでした。")

df = pd.concat(dfs, ignore_index=True)

log("=" * 70)
log("CSV結合完了")
log(f"有効CSV数: {len(dfs):,}")
log(f"スキップCSV数: {len(skipped_files):,}")
log(f"失敗CSV数: {len(failed_files):,}")
log(f"結合後行数: {len(df):,}")


# ==========================
# 型変換
# ==========================
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["change_point_count"] = pd.to_numeric(
    df["change_point_count"],
    errors="coerce"
).fillna(0).astype(int)

df = df.dropna(subset=["date"])
df = df.sort_values("date")

log(f"method数: {df['method'].nunique()}")
log(f"combo数: {df['combo'].nunique()}")
log(f"source_csv数: {df['source_csv'].nunique()}")


# ==========================
# 1. method別の日別集計
# ==========================
log("=" * 70)
log("method別の日別集計を開始")

method_daily = (
    df.groupby(["date", "method"], as_index=False)["change_point_count"]
      .sum()
      .rename(columns={"change_point_count": "method_change_point_count"})
)

method_daily["method"] = pd.Categorical(
    method_daily["method"],
    categories=METHOD_ORDER,
    ordered=True
)

method_daily = method_daily.sort_values(["method", "date"])

method_daily.to_csv(
    OUT_METHOD_CSV,
    index=False,
    encoding=SAVE_ENCODING
)

log(f"method別CSV保存完了: {OUT_METHOD_CSV}")


# ==========================
# 2. method別ピボット
# ==========================
pivot = (
    method_daily.pivot_table(
        index="date",
        columns="method",
        values="method_change_point_count",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

for method in METHOD_ORDER:
    if method not in pivot.columns:
        pivot[method] = 0

pivot = pivot[["date"] + METHOD_ORDER]
pivot = pivot.sort_values("date")


# ==========================
# 3. method 5種類の重ね描画
# ==========================
log("=" * 70)
log("method 5種類の重ね描画を開始")

plt.figure(figsize=FIGSIZE_METHOD_OVERLAY)

for method in METHOD_ORDER:
    plt.plot(
        pivot["date"],
        pivot[method],
        linewidth=2,
        label=method
    )

plt.title("Daily Change Point Counts by Method - All CSV")
plt.xlabel("Date")
plt.ylabel("Change Point Count")
plt.grid(True)

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    OUT_METHOD_OVERLAY_PNG,
    dpi=DPI,
    bbox_inches="tight"
)

log(f"method重ね描画PNG保存完了: {OUT_METHOD_OVERLAY_PNG}")

if SHOW_METHOD_OVERLAY:
    plt.show()

plt.close()


# ==========================
# 4. methodごとの個別描画
# ==========================
log("=" * 70)
log("methodごとの個別描画を開始")

for i, method in enumerate(METHOD_ORDER, start=1):

    log(f"[{i}/{len(METHOD_ORDER)}] method={method}")

    plt.figure(figsize=FIGSIZE_METHOD_SINGLE)

    plt.plot(
        pivot["date"],
        pivot[method],
        linewidth=2
    )

    plt.title(f"Daily Change Point Counts - method={method} - All CSV")
    plt.xlabel("Date")
    plt.ylabel("Change Point Count")
    plt.grid(True)

    plt.tight_layout()

    out_png = METHOD_PLOT_DIR / f"daily_change_point_method_{method}_all_csv.png"

    plt.savefig(out_png, dpi=DPI)

    log(f"保存完了: {out_png}")

    if SHOW_METHOD_SINGLE:
        plt.show()

    plt.close()


# ==========================
# 5. method別ランキング
# ==========================
log("=" * 70)
log("method別ランキングを作成")

method_ranking = []

for method in METHOD_ORDER:
    max_idx = pivot[method].idxmax()

    method_ranking.append({
        "method": method,
        "total_count": int(pivot[method].sum()),
        "max_count": int(pivot.loc[max_idx, method]),
        "max_date": pivot.loc[max_idx, "date"].strftime("%Y-%m-%d")
    })

method_ranking_df = pd.DataFrame(method_ranking)

method_ranking_df = method_ranking_df.sort_values(
    ["total_count", "max_count"],
    ascending=False
)

method_ranking_df.to_csv(
    OUT_METHOD_RANKING_CSV,
    index=False,
    encoding=SAVE_ENCODING
)

print()
print("=== method別ランキング ===")
print(method_ranking_df.to_string(index=False))

log(f"method別ランキングCSV保存完了: {OUT_METHOD_RANKING_CSV}")


# ==========================
# 6. 日付別総変化点ランキング
# ==========================
log("=" * 70)
log("日付別総変化点ランキングを作成")

date_total = (
    df.groupby("date", as_index=False)["change_point_count"]
      .sum()
      .rename(columns={"change_point_count": "total_change_point_count"})
)

date_total = date_total.sort_values(
    "total_change_point_count",
    ascending=False
)

date_total.to_csv(
    OUT_DATE_RANKING_CSV,
    index=False,
    encoding=SAVE_ENCODING
)

log(f"日付別ランキングCSV保存完了: {OUT_DATE_RANKING_CSV}")

print()
print(f"=== 日付別総変化点ランキング TOP{TOP_N_DATES} ===")
print(date_total.head(TOP_N_DATES).to_string(index=False))


# ==========================
# 7. 日付別ランキングTOP Nを棒グラフ化
# ==========================
log("=" * 70)
log(f"日付別ランキングTOP{TOP_N_DATES}の棒グラフを作成")

top_dates = date_total.head(TOP_N_DATES).copy()

top_dates["date_str"] = top_dates["date"].dt.strftime("%Y-%m-%d")

top_dates_plot = top_dates.sort_values(
    "total_change_point_count",
    ascending=True
)

plt.figure(figsize=FIGSIZE_RANKING)

plt.barh(
    top_dates_plot["date_str"],
    top_dates_plot["total_change_point_count"]
)

plt.title(f"Top {TOP_N_DATES} Dates by Total Change Point Count - All CSV")
plt.xlabel("Total Change Point Count")
plt.ylabel("Date")
plt.grid(axis="x")

plt.tight_layout()

plt.savefig(
    OUT_DATE_RANKING_TOP_PNG,
    dpi=DPI,
    bbox_inches="tight"
)

log(f"日付別ランキングPNG保存完了: {OUT_DATE_RANKING_TOP_PNG}")

if SHOW_DATE_RANKING:
    plt.show()

plt.close()


# ==========================
# 8. 日付ランキングTOP Nのmethod内訳CSV
# ==========================
log("=" * 70)
log(f"TOP{TOP_N_DATES}日付のmethod別内訳を作成")

top_date_set = set(top_dates["date"])

top_method_breakdown = method_daily[
    method_daily["date"].isin(top_date_set)
].copy()

top_method_breakdown["date_total"] = top_method_breakdown["date"].map(
    date_total.set_index("date")["total_change_point_count"]
)

top_method_breakdown = top_method_breakdown.sort_values(
    ["date_total", "date", "method"],
    ascending=[False, True, True]
)

OUT_TOP_METHOD_BREAKDOWN_CSV = OUTPUT_DIR / (
    f"daily_total_change_point_top{TOP_N_DATES}_method_breakdown_all_csv.csv"
)

top_method_breakdown.to_csv(
    OUT_TOP_METHOD_BREAKDOWN_CSV,
    index=False,
    encoding=SAVE_ENCODING
)

log(f"TOP日付のmethod別内訳CSV保存完了: {OUT_TOP_METHOD_BREAKDOWN_CSV}")


# ==========================
# 9. 読み込み対象CSV一覧を保存
# ==========================
log("=" * 70)
log("読み込みCSV一覧を保存")

read_log_df = pd.DataFrame({
    "loaded_csv": [p.name for p in csv_files if p.name not in skipped_files and p.name not in failed_files]
})

skip_log_df = pd.DataFrame({
    "skipped_csv": skipped_files
})

fail_log_df = pd.DataFrame({
    "failed_csv": failed_files
})

read_log_df.to_csv(
    OUTPUT_DIR / "loaded_csv_files_all_csv.csv",
    index=False,
    encoding=SAVE_ENCODING
)

skip_log_df.to_csv(
    OUTPUT_DIR / "skipped_csv_files_all_csv.csv",
    index=False,
    encoding=SAVE_ENCODING
)

fail_log_df.to_csv(
    OUTPUT_DIR / "failed_csv_files_all_csv.csv",
    index=False,
    encoding=SAVE_ENCODING
)


# ==========================
# 終了
# ==========================
log("=" * 70)
log("処理完了")

log(f"対象ディレクトリ: {BASE_DIR}")
log(f"有効CSV数: {len(dfs):,}")
log(f"結合後行数: {len(df):,}")

log(f"method別CSV: {OUT_METHOD_CSV}")
log(f"method重ね描画PNG: {OUT_METHOD_OVERLAY_PNG}")
log(f"method個別PNG保存先: {METHOD_PLOT_DIR}")
log(f"methodランキングCSV: {OUT_METHOD_RANKING_CSV}")
log(f"日付別ランキングCSV: {OUT_DATE_RANKING_CSV}")
log(f"日付別ランキングPNG: {OUT_DATE_RANKING_TOP_PNG}")
log(f"TOP日付method内訳CSV: {OUT_TOP_METHOD_BREAKDOWN_CSV}")

log(f"読み込みCSV一覧: {OUTPUT_DIR / 'loaded_csv_files_all_csv.csv'}")
log(f"スキップCSV一覧: {OUTPUT_DIR / 'skipped_csv_files_all_csv.csv'}")
log(f"失敗CSV一覧: {OUTPUT_DIR / 'failed_csv_files_all_csv.csv'}")

log(f"総処理時間: {time.time() - START_TIME:,.1f}s")